# 00 — Descubrimiento de Competidores

**Ejecutar UNA vez** antes de empezar la experimentacion.

Este notebook:
1. Pregunta las 15 queries fijas a ChatGPT y Gemini reales
2. Extrae las URLs que citan en sus respuestas
3. Rankea por frecuencia de aparicion
4. Permite revisar y aprobar la lista final
5. Scrapea las URLs seleccionadas
6. Genera embeddings y congela el vectorstore FAISS

**Resultado**: `data/frozen_vectorstore/` con el indice FAISS congelado

Ver ADR-004 y ADR-006 en `docs/DECISIONS.md`

In [ ]:
# === Setup del entorno ===
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

# Detectar si estamos en Kaggle o en local
if os.path.exists('/kaggle'):
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
    os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'
    sys.path.insert(0, '/kaggle/input/tfg')
else:
    from dotenv import load_dotenv
    load_dotenv()
    sys.path.insert(0, os.path.abspath('..'))

# Resolver PROJECT_ROOT para rutas de archivos (no depender del CWD)
from src.config import PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Cargar configuracion y queries ===
from src.config import load_experiment_config, get_all_queries

config = load_experiment_config()
queries = get_all_queries()

print(f"Target: {config['target_url']}")
print(f"Queries: {len(queries)}")
for i, q in enumerate(queries, 1):
    print(f"  {i:2d}. {q}")

In [ ]:
# === Descubrir competidores desde LLMs reales ===
from src.discovery.competitor_finder import CompetitorFinder

finder = CompetitorFinder(config)
results = finder.discover_competitors(queries, delay=2.0)

print(f"\nFecha: {results['discovery_date']}")
print(f"Engines: {results['engines_used']}")
print(f"\nTop competidores (por frecuencia):")
for entry in results['aggregated_urls'][:20]:
    print(f"  [{entry['frequency']}x] {entry['domain']} — {entry['url'][:80]}")

In [ ]:
# === Revisar y seleccionar competidores ===
# Editar esta lista manualmente si es necesario
selected_urls = results['top_competitors'][:10]

print("URLs seleccionadas para el vectorstore congelado:")
for i, url in enumerate(selected_urls, 1):
    print(f"  {i}. {url}")

# Guardar resultados del discovery (ruta absoluta via PROJECT_ROOT)
output_path = PROJECT_ROOT / 'config' / 'frozen_competitors.json'
finder.save_results(results, str(output_path))

In [ ]:
# === Scrape las URLs seleccionadas + target ===
from src.processing.html_processor import StructuredWebLoader
from src.processing.chunker import TokenAwareChunker

chunker = TokenAwareChunker(config)
all_urls = [config['target_url']] + selected_urls
all_docs = []

for url in all_urls:
    print(f"Scraping: {url}")
    loader = StructuredWebLoader(url)
    docs = loader.load()
    if docs:
        is_target = url == config['target_url']
        for d in docs:
            d.metadata['is_target'] = is_target
        chunks = chunker.chunk_documents(docs)
        all_docs.extend(chunks)
        print(f"  -> {len(chunks)} chunks")
    else:
        print(f"  -> FAILED")

print(f"\nTotal documentos: {len(all_docs)}")

In [ ]:
# === Generar embeddings y crear vectorstore FAISS ===
from src.processing.embeddings import create_embeddings
from langchain_community.vectorstores import FAISS

embeddings = create_embeddings(config)

# Separar docs de competidores (se congelan) del target (se regenera cada run)
competitor_docs = [d for d in all_docs if not d.metadata.get('is_target', False)]
target_docs = [d for d in all_docs if d.metadata.get('is_target', False)]

print(f"Competitor docs: {len(competitor_docs)}")
print(f"Target docs: {len(target_docs)}")

# Crear y guardar vectorstore de competidores (ruta absoluta)
if competitor_docs:
    competitor_vs = FAISS.from_documents(competitor_docs, embeddings)
    
    vs_path = PROJECT_ROOT / 'data' / 'frozen_vectorstore'
    vs_path.mkdir(parents=True, exist_ok=True)
    competitor_vs.save_local(str(vs_path))
    print(f"\nVectorstore congelado guardado en {vs_path}/")
else:
    print("ERROR: No hay documentos de competidores para congelar")

## Siguiente paso

El vectorstore esta congelado en `data/frozen_vectorstore/`. 
Ahora ejecuta `experimental_run.ipynb` para cada run experimental.

**IMPORTANTE**: No vuelvas a ejecutar este notebook a menos que quieras
cambiar el set de competidores (lo cual invalida todos los runs anteriores).